In [1]:
#locate OriC and DNA A boxes in Salmonella enterica genome using minimum skew and frequent 9-mers with mismatches + reverse complements. 

#1) Anonymous Functions

from collections import defaultdict

# -------------------------

# Define a function called reverse complement which takes a DNA pattern as input and returns its reverse complement.


def reverse_complement(pattern):

    comp = {'A':'T', 'T':'A', 'C':'G', 'G':'C'}

    return ''.join(comp[b] for b in reversed(pattern))

# -------------------------

# Define a function called Hamming distance which compares two strings of equal length and counts the number of positions at which the symbols are different.


def hamming_distance(p, q):

    return sum(1 for a, b in zip(p, q) if a != b)

# -------------------------

# Define a function called Neighbors which takes a DNA pattern and an integer d as input and returns the set of all k-mers that are at most d mismatches away from the pattern.


def neighbors(pattern, d):

    if d == 0:

        return {pattern}

    if len(pattern) == 1:

        return {'A', 'C', 'G', 'T'}

    suffix_neighbors = neighbors(pattern[1:], d) #recursive call on the suffix of the pattern

    neighborhood = set()

    for text in suffix_neighbors: #loop through the neighbors of the suffix

        if hamming_distance(pattern[1:], text) < d:

            for x in "ACGT":

                neighborhood.add(x + text)

        else:

            neighborhood.add(pattern[0] + text)

    return neighborhood

# -------------------------

# Define a function calledMinimum Skew which finds the positions where the GC skew is lowest. 


def minimum_skew(genome):

    skew = 0

    values = [0]

    for base in genome: #Loop through each nucleotide. NB: G increases skew, C decreases skew.

        if base == 'G':

            skew += 1

        elif base == 'C':

            skew -= 1

        values.append(skew)

    min_val = min(values)

    positions = [i for i, v in enumerate(values) if v == min_val] #Find all positions where skew is minimum.

    return positions

# -------------------------

# Define a function called frequent_words_with_mismatches_rc: Finds most frequent k-mers in DNA sequence allowing mismatches and reverse complements.


def frequent_words_with_mismatches_rc(text, k, d):

    freq = defaultdict(int) #Creates counting dictionary.

    for i in range(len(text) - k + 1): #Slide through sequence using windows of size k.

        pattern = text[i:i+k]

        for neighbor in neighbors(pattern, d): #Generate all variants within d mismatches.

            rc = reverse_complement(neighbor) #Compute reverse complement.

            freq[neighbor] += 1

            freq[rc] += 1

    max_count = max(freq.values())

    return sorted([p for p in freq if freq[p] == max_count]) #Return all k-mers with maximum count, sorted alphabetically.



# MAIN

# -------------------------

# Open the file containing the Salmonella enterica genome

with open("Salmonella_enterica.txt", "r") as f:

    lines = f.readlines()

genome = ''.join(line.strip() for line in lines[1:]) ## Ignore header


# STEP 1: minimum skew

positions = minimum_skew(genome)

print("Minimum skew positions:")

print(*positions)

k = 9

d = 1

window = 500

# STEP 2: Find likely oriC positions using Minimum Skew. 

for pos in positions:

    start = max(0, pos - window//2)

    end = min(len(genome), start + window)

    region = genome[start:end]

    # STEP 3: find frequent candidate DNaA boxes in the region around each minimum skew position using frequent_words_with_mismatches_rc function.
    
    motifs = frequent_words_with_mismatches_rc(region, k, d)

    print("\nCandidate ori region near position:", pos)

    print("Window:", start, "-", end)

    print("Most frequent candidate DnaA boxes:")

    print(" ".join(motifs))




Minimum skew positions:
3764856 3764858

Candidate ori region near position: 3764856
Window: 3764606 - 3765106
Most frequent candidate DnaA boxes:
AGCTTCCGG CCGGAAGCT

Candidate ori region near position: 3764858
Window: 3764608 - 3765108
Most frequent candidate DnaA boxes:
AGCTTCCGG CCGGAAGCT
